# 03 — Graph Construction
**Steps 14–16:** Convert valid pixels to graph nodes, build 8-connected spatial edges, and Z-score normalise node features.

In [ ]:
import os
import numpy as np
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler
from scipy.sparse import coo_matrix
from tqdm import tqdm

PROCESSED_DIR = os.path.join('..', 'data', 'processed')
OUTPUTS_DIR   = os.path.join('..', 'outputs')
os.makedirs(OUTPUTS_DIR, exist_ok=True)

In [ ]:
# ── Load preprocessed arrays ──────────────────────────────────
X     = np.load(os.path.join(PROCESSED_DIR, 'X_valid.npy'))          # (N, 8)
y     = np.load(os.path.join(PROCESSED_DIR, 'y_valid.npy'))          # (N,)
mask  = np.load(os.path.join(PROCESSED_DIR, 'valid_mask.npy'))       # (H, W) bool
shape = tuple(np.load(os.path.join(PROCESSED_DIR, 'spatial_shape.npy')))

N = X.shape[0]
print(f'Loaded {N:,} valid nodes | Features: {X.shape[1]} | Classes: {np.unique(y)}')

In [ ]:
# ── STEP 15: Build Spatial Edges (8-Connected Neighbourhood) ─

def build_edge_index(mask, shape):
    """
    Build COO edge index for all valid pixel pairs in an 8-connected grid.

    Returns:
        edge_index : torch.LongTensor shape (2, E)
    """
    H, W = shape

    # Map each (row, col) → valid node index
    node_idx = np.full((H, W), -1, dtype=np.int64)
    valid_positions = np.argwhere(mask)   # (N, 2)
    for i, (r, c) in enumerate(valid_positions):
        node_idx[r, c] = i

    # 8-connected offsets
    offsets = [(-1,-1), (-1,0), (-1,1),
               ( 0,-1),         ( 0,1),
               ( 1,-1), ( 1,0), ( 1,1)]

    src_list, dst_list = [], []

    for i, (r, c) in enumerate(tqdm(valid_positions, desc='Building edges')):
        for dr, dc in offsets:
            nr, nc = r + dr, c + dc
            if 0 <= nr < H and 0 <= nc < W and node_idx[nr, nc] >= 0:
                src_list.append(i)
                dst_list.append(node_idx[nr, nc])

    edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
    print(f'✅ Edges built: {edge_index.shape[1]:,} (bidirectional)')
    return edge_index


edge_index = build_edge_index(mask, shape)

In [ ]:
# ── STEP 16: Z-Score Normalisation ───────────────────────────

scaler = StandardScaler()
X_norm = scaler.fit_transform(X).astype(np.float32)

# Save scaler for inference use
import joblib
joblib.dump(scaler, os.path.join(PROCESSED_DIR, 'feature_scaler.pkl'))
print(f'✅ Features normalised. Mean: {X_norm.mean():.4f}, Std: {X_norm.std():.4f}')

In [ ]:
# ── STEP 17: Dataset Split ────────────────────────────────────

from sklearn.model_selection import train_test_split

TOTAL     = X_norm.shape[0]
TRAIN_SZ  = 1_460_689
VAL_SZ    = 313_005
TEST_SZ   = 313_005

indices = np.arange(TOTAL)
train_idx, temp_idx = train_test_split(indices, train_size=TRAIN_SZ, random_state=42)
val_idx, test_idx   = train_test_split(temp_idx, test_size=TEST_SZ,   random_state=42)

train_mask = torch.zeros(TOTAL, dtype=torch.bool)
val_mask   = torch.zeros(TOTAL, dtype=torch.bool)
test_mask  = torch.zeros(TOTAL, dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx]     = True
test_mask[test_idx]   = True

print(f'✅ Split — Train: {train_mask.sum():,} | Val: {val_mask.sum():,} | Test: {test_mask.sum():,}')

In [ ]:
# ── Build PyTorch Geometric Data Object ──────────────────────

graph_data = Data(
    x          = torch.tensor(X_norm, dtype=torch.float),
    edge_index = edge_index,
    y          = torch.tensor(y, dtype=torch.long),
    train_mask = train_mask,
    val_mask   = val_mask,
    test_mask  = test_mask
)

print(graph_data)

# Save graph
torch.save(graph_data, os.path.join(PROCESSED_DIR, 'graph_data.pt'))
print('✅ Graph saved to data/processed/graph_data.pt')